#PDC Assignment 2 -- Question 9
##Task-Graph Analyser



In [1]:
"""
PDC Assignment 2 - Question 9: Task-Graph Analyser
Run with: python q9_task_graph_analyser.py
"""
W = {"T1": 8, "T2": 6, "T3": 3, "T4": 5, "T5": 3,
     "T6": 7, "T7": 3, "T8": 5, "T9": 2}
PRED = {
    "T1": [], "T2": [], "T3": [],
    "T4": ["T1"], "T5": ["T1", "T2"], "T6": ["T2", "T3"],
    "T7": ["T4", "T5"], "T8": ["T6"], "T9": ["T7", "T8"]
}

def topological_order(weights, pred):
    remaining, done, order = set(weights), set(), []
    while remaining:
        ready = sorted(t for t in remaining if set(pred[t]) <= done)
        if not ready:
            raise ValueError("Cycle or invalid dependency in graph.")
        t = ready[0]
        order.append(t); done.add(t); remaining.remove(t)
    return order

def analyse(weights, pred):
    order = topological_order(weights, pred)
    succ = {t: [] for t in weights}
    for t, ps in pred.items():
        for p in ps:
            succ[p].append(t)
    starts = sorted(t for t in weights if not pred[t])
    finishes = sorted(t for t in weights if not succ[t])
    est, eft = {}, {}
    for t in order:
        est[t] = max((eft[p] for p in pred[t]), default=0)
        eft[t] = est[t] + weights[t]
    critical = max(eft[t] for t in finishes)
    paths = []
    def walk(t, path, length):
        path, length = path + [t], length + weights[t]
        if t in finishes:
            paths.append((path, length))
        else:
            for nxt in sorted(succ[t]):
                walk(nxt, path, length)
    for t in starts:
        walk(t, [], 0)
    events = sorted(set(est.values()) | set(eft.values()))
    maxc, intervals = 0, []
    for a, b in zip(events, events[1:]):
        count = sum(est[t] <= a and eft[t] > a for t in weights)
        if count > maxc:
            maxc, intervals = count, [(a, b)]
        elif count == maxc and count > 0:
            intervals.append((a, b))
    return order, succ, starts, finishes, est, eft, paths, critical, sum(weights.values()), maxc, intervals

def schedule(weights, pred, pcount):
    completed, running, rows = set(), {}, []
    now = 0
    while len(completed) < len(weights):
        for proc in sorted(list(running)):
            task, end = running[proc]
            if end == now:
                completed.add(task); del running[proc]
        for i in range(1, pcount + 1):
            proc = f"P{i}"
            if proc in running:
                continue
            active = {t for t, _ in running.values()}
            ready = sorted(t for t in weights if t not in completed and t not in active
                           and set(pred[t]) <= completed)
            if ready:
                t = ready[0]
                end = now + weights[t]
                running[proc] = (t, end)
                rows.append((proc, t, now, end))
        if len(completed) == len(weights):
            break
        if not running:
            raise ValueError("No ready task; graph may contain a cycle.")
        now = min(end for _, end in running.values())
    return rows, max(r[3] for r in rows)

def report(title, weights, pred):
    order, succ, starts, finishes, est, eft, paths, critical, work, maxc, intervals = analyse(weights, pred)
    print("\\n" + "="*60 + "\\n" + title + "\\n" + "="*60)
    print("Start nodes:", starts)
    print("Finish nodes:", finishes)
    print("\\nAll start-to-finish paths:")
    for path, length in paths:
        print(" -> ".join(path), "=", length)
    print("\\nCritical path(s):")
    for path, length in paths:
        if length == critical:
            print(" -> ".join(path), "=", length)
    print("Total work:", work)
    print("Critical-path length:", critical)
    print("Average degree of concurrency:", f"{work/critical:.2f}")
    print("\\nASAP start/finish times:")
    for t in order:
        print(f"{t}: start={est[t]}, finish={eft[t]}")
    print("Maximum concurrency:", maxc, "during interval(s):", intervals)
    print("\\nProcessor schedules (smallest-numbered ready task; P1 checked first):")
    for p in (1, 2, 3):
        rows, total = schedule(weights, pred, p)
        print(f"\\np={p}; total time={total}; speedup={work/total:.2f}")
        print(f'{"Processor":<12}{"Task":<8}{"Start":>8}{"End":>8}')
        for proc, task, start, end in rows:
            print(f"{proc:<12}{task:<8}{start:>8}{end:>8}")

if __name__ == "__main__":
    report("QUESTION 2 GRAPH", W, PRED)

    # Q3(a): ((Islamabad OR Multan) AND 2022) AND IT
    W_A = {"I":12, "M":12, "Y":12, "D":12, "OR":10, "AND1":18, "AND2":13}
    P_A = {"I":[], "M":[], "Y":[], "D":[], "OR":["I","M"],
           "AND1":["OR","Y"], "AND2":["AND1","D"]}
    report("QUESTION 3(a) GRAPH", W_A, P_A)

    # Q3(b): (IT AND 2022) AND (Islamabad OR Multan)
    W_B = {"I":12, "M":12, "Y":12, "D":12, "AND1":14, "OR":10, "AND2":14}
    P_B = {"I":[], "M":[], "Y":[], "D":[], "AND1":["D","Y"],
           "OR":["I","M"], "AND2":["AND1","OR"]}
    report("QUESTION 3(b) GRAPH", W_B, P_B)


\n============================================================\nQUESTION 2 GRAPH\n============================================================
Start nodes: ['T1', 'T2', 'T3']
Finish nodes: ['T9']
\nAll start-to-finish paths:
T1 -> T4 -> T7 -> T9 = 18
T1 -> T5 -> T7 -> T9 = 16
T2 -> T5 -> T7 -> T9 = 14
T2 -> T6 -> T8 -> T9 = 20
T3 -> T6 -> T8 -> T9 = 17
\nCritical path(s):
T2 -> T6 -> T8 -> T9 = 20
Total work: 42
Critical-path length: 20
Average degree of concurrency: 2.10
\nASAP start/finish times:
T1: start=0, finish=8
T2: start=0, finish=6
T3: start=0, finish=3
T4: start=8, finish=13
T5: start=8, finish=11
T6: start=6, finish=13
T7: start=13, finish=16
T8: start=13, finish=18
T9: start=18, finish=20
Maximum concurrency: 3 during interval(s): [(0, 3), (8, 11)]
\nProcessor schedules (smallest-numbered ready task; P1 checked first):
\np=1; total time=42; speedup=1.00
Processor   Task       Start     End
P1          T1             0       8
P1          T2             8      14
P1        